# Cardiac Segmentation — Mamba Integration Study
**Full Q1 Paper Pipeline — Google Colab Pro**

This notebook trains all models, runs parameter-matched baselines, evaluates on the test set,
computes EF metrics, runs explainability on ALL models, and generates publication-ready tables.

**Storage strategy**: Everything saves locally on Colab during training.
At the end of each session, only essential files (checkpoints, metrics, figures) are copied to Drive.

## Requirements
- Colab Pro with GPU (G4 Blackwell 96GB / A100 80GB / V100 16GB — adjust batch sizes)
- CAMUS dataset in Google Drive at `CAMUS_public/database_nifti/`
- **Blackwell (sm_120) GPUs**: Require PyTorch nightly with CUDA 12.8+

## Session Plan
| Session | What | Time (G4/A100) |
|---------|------|----------------|
| 1 | Base models (9) + Mamba models (9) | ~4-5h |
| 2 | Mamba2 (9) + VMamba (9) | ~6-8h |
| 3 | Param-matched (9) + Eval + EF + Benchmarks + Explainability | ~3-4h |

---
## 1. Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PyTorch nightly with CUDA 12.8 — required for Blackwell (sm_120) GPUs
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128 -q

In [ ]:
# Install mamba-ssm + causal-conv1d — must compile from source for sm_120
!pip uninstall -y mamba-ssm causal-conv1d 2>/dev/null
!pip install causal-conv1d --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q
# If pre-built wheel fails (wrong arch), uncomment to build from source:
# !TORCH_CUDA_ARCH_LIST="12.0" pip install git+https://github.com/state-spaces/mamba.git --no-build-isolation -q

In [ ]:
# Clone repo (always pull latest)
import os
if os.path.exists('/content/mamba'):
    !cd /content/mamba && git pull
else:
    !git clone https://github.com/Seraf00/mamba.git /content/mamba
%cd /content/mamba

In [ ]:
# Install remaining dependencies
!pip install uv -q
!uv pip install -r requirements.txt --quiet

In [ ]:
# Suppress noisy warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TRITON_F32_DEFAULT'] = 'ieee'
import warnings
warnings.filterwarnings('ignore')

## 2. Copy CAMUS Dataset

In [ ]:
source_path = '/content/drive/MyDrive/CAMUS_public/database_nifti/'
destination_path = '/content/mamba/data/CAMUS'
os.makedirs(destination_path, exist_ok=True)

if len(os.listdir(destination_path)) < 100:
    print('Copying CAMUS dataset... (5-10 min)')
    !rsync -ah --info=progress2 "{source_path}" "{destination_path}"
    print('Done!')
else:
    print(f'CAMUS dataset already present ({len(os.listdir(destination_path))} items)')

## 3. Verify Setup

In [ ]:
!python scripts/check_mamba_setup.py

In [ ]:
%load_ext tensorboard

## 4. Save-to-Drive Helper
Only copies essential files to Drive — skips TensorBoard logs and other large temp files.

In [ ]:
import subprocess, glob

DRIVE_RESULTS = '/content/drive/MyDrive/Paper1/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

def save_to_drive():
    """Copy only essential files to Drive: checkpoints, metrics, figures.
    Skips TensorBoard logs, intermediate checkpoints, and other large temp files."""
    !rsync -ah --info=progress2 \
        --include='*/' \
        --include='best_model.pth' \
        --include='results.json' \
        --include='cv_results.json' \
        --include='all_results.json' \
        --include='summary.csv' \
        --include='training_log.csv' \
        --include='experiment_config.json' \
        --include='evaluation_results.json' \
        --include='results_table.tex' \
        --include='results.csv' \
        --include='*.png' \
        --include='*.json' \
        --include='*.csv' \
        --include='*.tex' \
        --include='*.txt' \
        --exclude='*' \
        /content/results/ "{DRIVE_RESULTS}/"
    
    # Print disk usage summary
    local_size = subprocess.getoutput('du -sh /content/results/ 2>/dev/null').split()[0]
    drive_size = subprocess.getoutput(f'du -sh "{DRIVE_RESULTS}/" 2>/dev/null').split()[0]
    print(f'\nLocal: {local_size} | Drive: {drive_size}')
    print(f'Saved to {DRIVE_RESULTS}')

---
# SESSION 1: Base + Mamba Models

## 5. Train Base Models (9 models, ~1-2h on A100)

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 128 \
    --early_stopping 20 \
    --num_workers 4 \
    --base_only \
    --mixed_precision \
    --skip_benchmark \
    --exp_name base_models \
    --output_dir /content/results/

In [ ]:
# Clean intermediate checkpoints to save disk space
!find /content/results -name "checkpoint_epoch_*.pth" -delete
!df -h /content

## 6. Train Mamba-Enhanced Models (9 models, ~2-3h on A100)
Uses CUDA-optimized selective scan (~1-2s/batch).

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 16 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants mamba \
    --mixed_precision \
    --skip_benchmark \
    --exp_name mamba_models \
    --output_dir /content/results/

In [ ]:
# Clean intermediate checkpoints to save disk space
!find /content/results -name "checkpoint_epoch_*.pth" -delete
!df -h /content

## Save Session 1 to Drive

In [ ]:
save_to_drive()

---
# SESSION 2: Mamba2 + VMamba Models
Run setup cells 1-4 first if starting a new session.

## 7. Train Mamba2-Enhanced Models (9 models, ~3-4h)
Uses SSD algorithm with Triton kernels.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 16 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants mamba2 \
    --mixed_precision \
    --skip_benchmark \
    --exp_name mamba2_models \
    --output_dir /content/results/

In [ ]:
# Clean intermediate checkpoints to save disk space
!find /content/results -name "checkpoint_epoch_*.pth" -delete
!df -h /content

## 8. Train VMamba-Enhanced Models (9 models, ~4-6h)
Uses 4-directional cross-scan. AMP dtype fix ensures CUDA kernel works.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 16 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants vmamba \
    --mixed_precision \
    --skip_benchmark \
    --exp_name vmamba_models \
    --output_dir /content/results/

In [ ]:
# Clean intermediate checkpoints to save disk space
!find /content/results -name "checkpoint_epoch_*.pth" -delete
!df -h /content

## Save Session 2 to Drive

In [ ]:
save_to_drive()

---
# SESSION 3: Parameter-Matched + Evaluation + Explainability
Run setup cells 1-4 first. Then restore previous checkpoints from Drive:

In [ ]:
# Restore previous session checkpoints from Drive (needed for evaluation + explainability)
!rsync -ah --info=progress2 "{DRIVE_RESULTS}/" /content/results/

## 9. Compute Parameter-Matched Base Features
Finds wider base models that match Mamba param counts for fair comparison.
A reviewer will ask: *Is the improvement from Mamba or from more parameters?*

In [ ]:
!python scripts/param_match.py --output_json /content/results/param_config.json

## 10. Train Parameter-Matched Wider Base Models (~2-3h)

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 8 \
    --early_stopping 20 \
    --num_workers 4 \
    --base_only \
    --param_matched \
    --param_config /content/results/param_config.json \
    --mixed_precision \
    --skip_benchmark \
    --exp_name param_matched \
    --output_dir /content/results/

In [ ]:
# Clean intermediate checkpoints to save disk space
!find /content/results -name "checkpoint_epoch_*.pth" -delete
!df -h /content

## 11. Efficiency Benchmarks
Measures FLOPs, latency, memory, and parameter counts for all models.

In [ ]:
!python scripts/benchmark.py \
    --all \
    --mamba_types mamba mamba2 vmamba \
    --output /content/results/benchmark_efficiency.csv

## 12. Evaluate All Models on Test Set
Computes Dice, IoU, HD95, ASD + biplane EF with pixel spacing.
Generates LaTeX tables and statistical tests (Wilcoxon + 95% CI).

In [ ]:
exp_dirs = [
    '/content/results/base_models',
    '/content/results/mamba_models',
    '/content/results/mamba2_models',
    '/content/results/vmamba_models',
    '/content/results/param_matched',
]

for exp_dir in exp_dirs:
    if os.path.exists(exp_dir):
        print(f'\n{"="*70}')
        print(f'Evaluating: {exp_dir}')
        print(f'{"="*70}')
        !python scripts/evaluate_all_models.py \
            --checkpoint_dir "{exp_dir}" \
            --data_dir /content/mamba/data/CAMUS/ \
            --compute_ef \
            --split test \
            --batch_size 8

---
## 13. Explainability: ALL Models — Grad-CAM + Uncertainty Comparison
Runs Grad-CAM and MC Dropout uncertainty on **every** trained model.
Generates comparison grid figures (rows=models, cols=test samples).

This shows how each architecture makes decisions from the same input image.

In [ ]:
!python scripts/explain_all_models.py \
    --results_dirs \
        /content/results/base_models \
        /content/results/mamba_models \
        /content/results/mamba2_models \
        /content/results/vmamba_models \
        /content/results/param_matched \
    --data_dir /content/mamba/data/CAMUS/ \
    --methods gradcam uncertainty mamba_states \
    --n_samples 3 \
    --n_mc_samples 20 \
    --target_class 1 \
    --output /content/results/explainability/

## 14. Explainability: Grad-CAM for All 3 Classes (LV, MYO, LA)
Generates separate comparison grids for each cardiac structure.

In [ ]:
# Myocardium (class 2)
!python scripts/explain_all_models.py \
    --results_dirs \
        /content/results/base_models \
        /content/results/mamba_models \
    --data_dir /content/mamba/data/CAMUS/ \
    --methods gradcam \
    --n_samples 3 \
    --target_class 2 \
    --output /content/results/explainability_myo/

# Left atrium (class 3)
!python scripts/explain_all_models.py \
    --results_dirs \
        /content/results/base_models \
        /content/results/mamba_models \
    --data_dir /content/mamba/data/CAMUS/ \
    --methods gradcam \
    --n_samples 3 \
    --target_class 3 \
    --output /content/results/explainability_la/

## 15. Explainability: Clinical Reports (Best Models)
Generates structured clinical reports for the best base and Mamba models.

In [ ]:
# Clinical report for best Mamba model (change path to your best model)
!python scripts/explain.py \
    --checkpoint /content/results/mamba_models/mamba_unet_v1_mamba/best_model.pth \
    --model mamba_unet_v1 \
    --data_dir /content/mamba/data/CAMUS/ \
    --methods all \
    --clinical_report \
    --n_samples 20 \
    --output /content/results/explainability/clinical_report/

## 16. Save Everything to Drive
Copies only essential files (checkpoints, metrics, figures, tables) — skips TensorBoard logs.

In [ ]:
save_to_drive()
print('\nAll done! Results are in Google Drive at:', DRIVE_RESULTS)
print('\nWhat was saved:')
print('  - best_model.pth per model    (weights only, no optimizer state)')
print('  - results.json per model      (training metrics)')
print('  - training_log.csv per model  (epoch-by-epoch curves)')
print('  - evaluation_results.json     (test set Dice, HD95, EF)')
print('  - results_table.tex           (LaTeX table for paper)')
print('  - benchmark_efficiency.csv    (FLOPs, latency, memory)')
print('  - explainability/*.png        (Grad-CAM, uncertainty grids)')
print('\nWhat was NOT saved (to save space):')
print('  - TensorBoard logs')
print('  - __pycache__')

## TensorBoard (optional — run anytime during training)

In [ ]:
%tensorboard --logdir /content/results/

---
## Recovery: Resume After Session Timeout
If Colab disconnects mid-training, use `--resume_from` to skip completed models.

In [ ]:
# 1. Restore previous results from Drive
# !rsync -ah "{DRIVE_RESULTS}/" /content/results/

# 2. Resume from the model that was interrupted
# !python scripts/train_all_models.py \
#     --data_dir /content/mamba/data/CAMUS/ \
#     --epochs 100 --batch_size 8 --early_stopping 20 \
#     --num_workers 4 --mamba_only --mamba_variants mamba \
#     --mixed_precision --skip_benchmark \
#     --resume_from mamba_fpn_mamba \
#     --exp_name mamba_models \
#     --output_dir /content/results/